In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision.utils as vutils

SEED = 99

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if hasattr(torch, 'mps'):
    torch.mps.manual_seed(SEED)

IS_KAGGLE = os.path.exists('/kaggle/working')

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print('device:', DEVICE, '| kaggle:', IS_KAGGLE)


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchmetrics'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scipy', 'lightning-utilities', 'kagglehub'], check=True)
print("Done. ")


## Данные — CelebA

In [ ]:
import kagglehub

celeba_path = kagglehub.dataset_download('jessicali9530/celeba-dataset')
print('Downloaded to:', celeba_path)

In [ ]:
raw_imgs_dir = Path(celeba_path) / 'img_align_celeba' / 'img_align_celeba'

attr_df = pd.read_csv(Path(celeba_path) / 'list_attr_celeba.csv')
part_df = pd.read_csv(Path(celeba_path) / 'list_eval_partition.csv')

full_df = part_df.merge(attr_df[['image_id', 'Male']], on='image_id')
full_df['split'] = full_df['partition'].map({0: 'train', 1: 'val', 2: 'test'})
full_df['male'] = (full_df['Male'] == 1).astype(int)
full_df.drop(columns=['Male'], inplace=True)
full_df['src_path'] = full_df['image_id'].apply(lambda x: str(raw_imgs_dir / x))

In [ ]:
import numpy as np
from PIL import Image, ImageOps
from tqdm.auto import tqdm


def pick_best_face(boxes):
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return int(np.argmax(areas))


def pad_and_crop_face(pil_img, box, target_size=128, scale=1.35):
    W, H = pil_img.size
    x1, y1, x2, y2 = map(float, box)
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max((x2 - x1) * scale, (y2 - y1) * scale)
    nx1, ny1 = cx - side / 2, cy - side / 2
    nx2, ny2 = cx + side / 2, cy + side / 2
    pad = (max(0, int(np.floor(-nx1))), max(0, int(np.floor(-ny1))),
           max(0, int(np.ceil(nx2 - W))), max(0, int(np.ceil(ny2 - H))))
    if any(pad):
        pil_img = ImageOps.expand(pil_img, border=pad, fill=(0, 0, 0))
        nx1 += pad[0]; nx2 += pad[0]; ny1 += pad[1]; ny2 += pad[1]
    coords = tuple(map(lambda v: int(round(v)), (nx1, ny1, nx2, ny2)))
    return pil_img.crop(coords).resize((target_size, target_size), Image.BILINEAR)


def prepare_faces(out_root: Path, df: pd.DataFrame, img_size=128):
    if out_root.exists():
        print('Already preprocessed.')
        return

    for sp in ('train', 'val', 'test'):
        (out_root / sp).mkdir(parents=True)

    if IS_KAGGLE:
        # CelebA уже выровнена — просто resize
        ok, fail = 0, 0
        rows = []
        for i in tqdm(range(len(df)), desc='resize'):
            row = df.loc[i]
            src = Path(row['src_path'])
            if not src.exists():
                fail += 1; continue
            face = Image.open(src).convert('RGB').resize((img_size, img_size), Image.BILINEAR)
            out_path = out_root / row['split'] / row['image_id']
            face.save(out_path, quality=95)
            rows.append({'image_id': row['image_id'], 'split': row['split'],
                         'male': int(row['male']), 'out_path': str(out_path)})
            ok += 1
        pd.DataFrame(rows).to_csv(out_root / 'meta.csv', index=False)
        print(f'OK: {ok}, failed: {fail}')
    else:
        from facenet_pytorch import MTCNN
        detector = MTCNN(image_size=img_size, margin=20, min_face_size=40,
                         post_process=False, keep_all=False, device='cpu')
        ok, fail = 0, 0
        rows = []
        for i in tqdm(range(len(df))):
            row = df.loc[i]
            src = Path(row['src_path'])
            if not src.exists():
                fail += 1; continue
            img = Image.open(src).convert('RGB')
            with torch.inference_mode():
                boxes, _ = detector.detect(img)
            if boxes is None or len(boxes) == 0:
                fail += 1; continue
            boxes = np.asarray(boxes, float)
            face = pad_and_crop_face(img, boxes[pick_best_face(boxes)], target_size=img_size)
            out_path = out_root / row['split'] / row['image_id']
            face.save(out_path, quality=95)
            rows.append({'image_id': row['image_id'], 'split': row['split'],
                         'male': int(row['male']), 'out_path': str(out_path)})
            ok += 1
        pd.DataFrame(rows).to_csv(out_root / 'meta.csv', index=False)
        print(f'OK: {ok}, failed: {fail}')


In [ ]:
FACES_ROOT = Path('/kaggle/working/faces_128') if IS_KAGGLE else Path('faces_128')
N_SAMPLES = 10000 if IS_KAGGLE else 50000

sub_df = full_df.sample(N_SAMPLES, random_state=SEED).reset_index(drop=True)
prepare_faces(FACES_ROOT, sub_df, img_size=128)

meta = pd.read_csv(FACES_ROOT / 'meta.csv')
train_meta = meta[meta['split'] == 'train']
val_meta = meta[meta['split'] == 'val']
print(f'Train: {len(train_meta)}, Val: {len(val_meta)}')


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image


class FaceDataset(Dataset):
    def __init__(self, meta_df, transform, with_label=False):
        self.records = meta_df.reset_index(drop=True)
        self.transform = transform
        self.with_label = with_label

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.loc[idx]
        img = Image.open(row['out_path']).convert('RGB')
        img = self.transform(img)
        if self.with_label:
            label = torch.tensor(int(row['male']), dtype=torch.long)
            return img, label
        return img


def _pil_to_tensor(pic):
    ch = len(pic.getbands())
    buf = bytearray(pic.tobytes())
    t = torch.frombuffer(buf, dtype=torch.uint8).clone()
    return t.view(pic.size[1], pic.size[0], ch).permute(2, 0, 1).float().div(255.0)

img_transform = T.Compose([
    T.Lambda(_pil_to_tensor),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

BS = 64
NW = 0 if IS_KAGGLE else 8

train_ds = FaceDataset(train_meta, transform=img_transform, with_label=False)
val_ds = FaceDataset(val_meta, transform=img_transform, with_label=False)

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,
                          num_workers=NW, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW)


In [ ]:
Z_DIM = 128
NGF = 64
NDF = 64


def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


class GenBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=4, s=2, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM, base=NGF, out_ch=3, n_classes=None, emb_dim=32):
        super().__init__()
        self.n_classes = n_classes
        if n_classes is not None:
            self.class_emb = nn.Embedding(n_classes, emb_dim)
            in_dim = z_dim + emb_dim
        else:
            in_dim = z_dim

        self.blocks = nn.Sequential(
            GenBlock(in_dim, base * 16, k=4, s=1, p=0),
            GenBlock(base * 16, base * 8),
            GenBlock(base * 8, base * 4),
            GenBlock(base * 4, base * 2),
            GenBlock(base * 2, base),
            nn.ConvTranspose2d(base, out_ch, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z, y=None):
        if self.n_classes is not None:
            emb = self.class_emb(y).unsqueeze(-1).unsqueeze(-1)
            z = torch.cat([z, emb], dim=1)
        return self.blocks(z)


class DiscBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if norm:
            g = min(32, out_ch)
            layers.append(nn.GroupNorm(g, out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self, base=NDF, in_ch=3, n_classes=None):
        super().__init__()
        self.n_classes = n_classes
        feat = base * 16

        self.feature_net = nn.Sequential(
            DiscBlock(in_ch, base, norm=False),
            DiscBlock(base, base * 2),
            DiscBlock(base * 2, base * 4),
            DiscBlock(base * 4, base * 8),
            DiscBlock(base * 8, feat),
        )
        self.head = nn.Conv2d(feat, 1, 4, 1, 0, bias=False)
        self.proj = nn.Embedding(n_classes, feat) if n_classes else None
        self.feat_dim = feat

    def forward(self, x, y=None):
        feats = self.feature_net(x)
        score = self.head(feats).view(x.size(0))
        if self.proj is not None:
            fv = feats.sum(dim=(2, 3))
            score = score + (self.proj(y) * fv).sum(dim=1)
        return score

In [ ]:
gen = Generator(z_dim=Z_DIM, base=NGF).to(DEVICE)
disc = Discriminator(base=NDF).to(DEVICE)
gen.apply(init_weights)
disc.apply(init_weights)

print('Gen params:', sum(p.numel() for p in gen.parameters()))
print('Disc params:', sum(p.numel() for p in disc.parameters()))


In [ ]:
from torch import optim

LR_G = 2e-4
LR_D = 1e-4
BETAS = (0.0, 0.9)
LAMBDA_GP = 10.0
N_DISC = 1
TOTAL_EPOCHS = 25
GP_EVERY = 4

opt_g = optim.Adam(gen.parameters(), lr=LR_G, betas=BETAS)
opt_d = optim.Adam(disc.parameters(), lr=LR_D, betas=BETAS)

fixed_noise = torch.randn(64, Z_DIM, 1, 1, device=DEVICE)

CKPT_ROOT = Path('/kaggle/working/checkpoints') if IS_KAGGLE else Path('checkpoints')
SAMPLES_ROOT = Path('/kaggle/working/samples') if IS_KAGGLE else Path('samples')
(CKPT_ROOT / 'uncond').mkdir(parents=True, exist_ok=True)
(SAMPLES_ROOT / 'uncond').mkdir(parents=True, exist_ok=True)

ckpt_path = CKPT_ROOT / 'uncond' / 'last.pt'


In [ ]:
def grad_penalty(disc, real_imgs, fake_imgs, y=None, lam=10.0):
    b = real_imgs.size(0)
    alpha = torch.rand(b, 1, 1, 1, device=DEVICE)
    interp = (alpha * real_imgs + (1 - alpha) * fake_imgs).requires_grad_(True)
    d_interp = disc(interp, y) if y is not None else disc(interp)
    grads = torch.autograd.grad(
        outputs=d_interp, inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True, retain_graph=True,
    )[0]
    grads = grads.view(b, -1)
    penalty = ((grads.norm(2, dim=1) - 1) ** 2).mean() * lam
    return penalty


def _tensor_to_pil(grid):
    g = grid.mul(255).clamp(0, 255).byte()
    c, h, w = g.shape
    raw = bytes(bytearray(g.permute(1, 2, 0).reshape(-1).tolist()))
    return Image.frombytes('RGB', (w, h), raw)

def save_grid(generator, noise, path, y=None):
    generator.eval()
    with torch.inference_mode():
        fake = generator(noise, y).cpu()
        fake = (fake * 0.5 + 0.5).clamp(0, 1)
        _tensor_to_pil(vutils.make_grid(fake, nrow=8)).save(str(path))
    generator.train()


def move_optimizer(opt, device):
    for state in opt.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(device)

In [ ]:
hist = {'d': [], 'g': [], 'wass': [], 'gp': []}
epoch_metrics = {'epoch': [], 'fid': [], 'is_mean': [], 'is_std': []}
step = 0
start_epoch = 1

if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location='cpu')
    gen.load_state_dict(ckpt['gen'])
    disc.load_state_dict(ckpt['disc'])
    opt_g.load_state_dict(ckpt['opt_g'])
    opt_d.load_state_dict(ckpt['opt_d'])
    hist = ckpt.get('hist', hist)
    epoch_metrics = ckpt.get('epoch_metrics', epoch_metrics)
    step = ckpt.get('step', 0)
    start_epoch = ckpt.get('epoch', 0) + 1
    print(f'Resumed from epoch {start_epoch - 1}, step {step}')
    move_optimizer(opt_g, DEVICE)
    move_optimizer(opt_d, DEVICE)

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore


def clip_01(t):
    return (t.clamp(-1, 1) + 1) / 2


FID_STATE = CKPT_ROOT / 'fid_real.pt'

fid_m = FrechetInceptionDistance(feature=2048, normalize=True).to('cpu')
is_m = InceptionScore(normalize=True, splits=10).to('cpu')


def cache_real_stats(loader, fid_metric, max_n=128):
    fid_metric.reset()
    n = 0
    for batch in loader:
        imgs = batch[0] if isinstance(batch, (tuple, list)) else batch
        fid_metric.update(clip_01(imgs).to('cpu', torch.float32), real=True)
        n += imgs.size(0)
        if n >= max_n:
            break
    n_real = int(fid_metric.real_features_num_samples.item()
                 if torch.is_tensor(fid_metric.real_features_num_samples)
                 else fid_metric.real_features_num_samples)
    torch.save({'n_real': n_real,
                'sum': fid_metric.real_features_sum.cpu(),
                'cov_sum': fid_metric.real_features_cov_sum.cpu(),
                'n_samples': (fid_metric.real_features_num_samples.cpu()
                              if torch.is_tensor(fid_metric.real_features_num_samples)
                              else torch.tensor(n_real, dtype=torch.long))}, FID_STATE)
    return n_real


@torch.inference_mode()
def eval_metrics(generator, fid_metric, is_metric, n_fake=128):
    generator.eval()
    fid_metric.reset()
    is_metric.reset()

    state = torch.load(FID_STATE, map_location='cpu')
    n_real = int(state['n_real'])
    fid_metric.real_features_sum.copy_(state['sum'])
    fid_metric.real_features_cov_sum.copy_(state['cov_sum'])
    if torch.is_tensor(fid_metric.real_features_num_samples):
        fid_metric.real_features_num_samples.copy_(state['n_samples'])
    else:
        fid_metric.real_features_num_samples = int(state['n_samples'].item())

    generated = 0
    while generated < n_real:
        b = min(64, n_real - generated)
        z = torch.randn(b, Z_DIM, 1, 1, device=DEVICE)
        fake = clip_01(generator(z)).cpu()
        fid_metric.update(fake, real=False)
        is_metric.update(fake)
        generated += b

    fid = float(fid_metric.compute().item())
    is_mean, is_std = is_metric.compute()
    generator.train()
    return fid, float(is_mean.item()), float(is_std.item()), n_real


if not FID_STATE.exists():
    cache_real_stats(val_loader, fid_m, max_n=128)

In [ ]:
for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    for real in train_loader:
        step += 1
        real = real.to(DEVICE)
        b = real.size(0)

        for _ in range(N_DISC):
            z = torch.randn(b, Z_DIM, 1, 1, device=DEVICE)
            fake = gen(z).detach()

            d_real = disc(real).mean()
            d_fake = disc(fake).mean()
            wasserstein = d_real - d_fake

            gp = (grad_penalty(disc, real, fake, lam=LAMBDA_GP * GP_EVERY)
                  if step % GP_EVERY == 0
                  else torch.zeros((), device=DEVICE))

            loss_d = -wasserstein + gp
            opt_d.zero_grad(set_to_none=True)
            loss_d.backward()
            opt_d.step()

        z = torch.randn(b, Z_DIM, 1, 1, device=DEVICE)
        loss_g = -disc(gen(z)).mean()
        opt_g.zero_grad(set_to_none=True)
        loss_g.backward()
        opt_g.step()

        hist['d'].append(loss_d.item())
        hist['g'].append(loss_g.item())
        hist['wass'].append(wasserstein.item())
        hist['gp'].append(gp.item())

        if step % 300 == 0:
            save_grid(gen, fixed_noise, SAMPLES_ROOT / 'uncond' / f'step_{step:07d}.png')
            print(f'e{epoch} s{step} | D={loss_d.item():.3f} G={loss_g.item():.3f} W={wasserstein.item():.3f}')

    fid, is_mean, is_std, n = eval_metrics(gen, fid_m, is_m)
    epoch_metrics['epoch'].append(epoch)
    epoch_metrics['fid'].append(fid)
    epoch_metrics['is_mean'].append(is_mean)
    epoch_metrics['is_std'].append(is_std)
    print(f'[uncond] epoch {epoch} FID={fid:.2f} IS={is_mean:.2f}±{is_std:.2f} n={n}')

    save_grid(gen, fixed_noise, SAMPLES_ROOT / 'uncond' / f'epoch_{epoch:03d}.png')
    tmp = ckpt_path.with_suffix('.tmp')
    torch.save({'epoch': epoch, 'step': step, 'gen': gen.state_dict(),
                'disc': disc.state_dict(), 'opt_g': opt_g.state_dict(),
                'opt_d': opt_d.state_dict(), 'hist': hist,
                'epoch_metrics': epoch_metrics}, tmp)
    tmp.replace(ckpt_path)

In [ ]:
cond_train_ds = FaceDataset(train_meta, transform=img_transform, with_label=True)
cond_val_ds = FaceDataset(val_meta, transform=img_transform, with_label=True)

cond_train_loader = DataLoader(cond_train_ds, batch_size=BS, shuffle=True,
                                num_workers=NW, drop_last=True)
cond_val_loader = DataLoader(cond_val_ds, batch_size=BS, shuffle=False, num_workers=NW)


In [ ]:
cgen = Generator(z_dim=Z_DIM, base=NGF, n_classes=2, emb_dim=32).to(DEVICE)
cdisc = Discriminator(base=NDF, n_classes=2).to(DEVICE)
cgen.apply(init_weights)
cdisc.apply(init_weights)

copt_g = optim.Adam(cgen.parameters(), lr=LR_G, betas=BETAS)
copt_d = optim.Adam(cdisc.parameters(), lr=LR_D, betas=BETAS)

fixed_y = torch.tensor([0] * 32 + [1] * 32, device=DEVICE)

(CKPT_ROOT / 'cond').mkdir(parents=True, exist_ok=True)
(SAMPLES_ROOT / 'cond').mkdir(parents=True, exist_ok=True)
cckpt_path = CKPT_ROOT / 'cond' / 'last.pt'


In [ ]:
FID_STATE_COND = CKPT_ROOT / 'fid_real_cond.pt'

cfid_m = FrechetInceptionDistance(feature=2048, normalize=True).to('cpu')
cis_m = InceptionScore(normalize=True, splits=10).to('cpu')

if not FID_STATE_COND.exists():
    # reuse same function but from cond_val_loader
    cfid_m.reset()
    n = 0
    for imgs, _ in cond_val_loader:
        cfid_m.update(clip_01(imgs).to('cpu', torch.float32), real=True)
        n += imgs.size(0)
        if n >= 256:
            break
    n_real_c = int(cfid_m.real_features_num_samples.item()
                   if torch.is_tensor(cfid_m.real_features_num_samples)
                   else cfid_m.real_features_num_samples)
    torch.save({'n_real': n_real_c,
                'sum': cfid_m.real_features_sum.cpu(),
                'cov_sum': cfid_m.real_features_cov_sum.cpu(),
                'n_samples': (cfid_m.real_features_num_samples.cpu()
                              if torch.is_tensor(cfid_m.real_features_num_samples)
                              else torch.tensor(n_real_c, dtype=torch.long))}, FID_STATE_COND)
    print(f'Cached cond FID real stats: n={n_real_c}')

In [ ]:
@torch.inference_mode()
def eval_metrics_cond(generator, fid_metric, is_metric, n_fake=256):
    generator.eval()
    fid_metric.reset()
    is_metric.reset()

    state = torch.load(FID_STATE_COND, map_location='cpu')
    n_real = int(state['n_real'])
    fid_metric.real_features_sum.copy_(state['sum'])
    fid_metric.real_features_cov_sum.copy_(state['cov_sum'])
    if torch.is_tensor(fid_metric.real_features_num_samples):
        fid_metric.real_features_num_samples.copy_(state['n_samples'])
    else:
        fid_metric.real_features_num_samples = int(state['n_samples'].item())

    generated = 0
    while generated < n_real:
        b = min(64, n_real - generated)
        z = torch.randn(b, Z_DIM, 1, 1, device=DEVICE)
        y_fake = torch.randint(0, 2, (b,), device=DEVICE)
        fake = clip_01(generator(z, y_fake)).cpu()
        fid_metric.update(fake, real=False)
        is_metric.update(fake)
        generated += b

    fid = float(fid_metric.compute().item())
    is_mean, is_std = is_metric.compute()
    generator.train()
    return fid, float(is_mean.item()), float(is_std.item()), n_real

In [ ]:
chist = {'d': [], 'g': [], 'wass': [], 'gp': []}
cepoch_metrics = {'epoch': [], 'fid': [], 'is_mean': [], 'is_std': []}
cstep = 0
cstart_epoch = 1

if cckpt_path.exists():
    ckpt = torch.load(cckpt_path, map_location='cpu')
    cgen.load_state_dict(ckpt['gen'])
    cdisc.load_state_dict(ckpt['disc'])
    copt_g.load_state_dict(ckpt['opt_g'])
    copt_d.load_state_dict(ckpt['opt_d'])
    chist = ckpt.get('hist', chist)
    cepoch_metrics = ckpt.get('epoch_metrics', cepoch_metrics)
    cstep = ckpt.get('step', 0)
    cstart_epoch = ckpt.get('epoch', 0) + 1
    print(f'Cond resumed from epoch {cstart_epoch - 1}')
    move_optimizer(copt_g, DEVICE)
    move_optimizer(copt_d, DEVICE)

In [ ]:
COND_EPOCHS = 20

for epoch in range(cstart_epoch, COND_EPOCHS + 1):
    for real, y in cond_train_loader:
        cstep += 1
        real = real.to(DEVICE)
        y = y.to(DEVICE)
        b = real.size(0)

        for _ in range(N_DISC):
            z = torch.randn(b, Z_DIM, 1, 1, device=DEVICE)
            fake = cgen(z, y).detach()

            d_real = cdisc(real, y).mean()
            d_fake = cdisc(fake, y).mean()
            wasserstein = d_real - d_fake

            gp = (grad_penalty(cdisc, real, fake, y=y, lam=LAMBDA_GP * GP_EVERY)
                  if cstep % GP_EVERY == 0
                  else torch.zeros((), device=DEVICE))

            loss_d = -wasserstein + gp
            copt_d.zero_grad(set_to_none=True)
            loss_d.backward()
            copt_d.step()

        z = torch.randn(b, Z_DIM, 1, 1, device=DEVICE)
        y_fake = torch.randint(0, 2, (b,), device=DEVICE)
        loss_g = -cdisc(cgen(z, y_fake), y_fake).mean()
        copt_g.zero_grad(set_to_none=True)
        loss_g.backward()
        copt_g.step()

        chist['d'].append(loss_d.item())
        chist['g'].append(loss_g.item())
        chist['wass'].append(wasserstein.item())
        chist['gp'].append(gp.item())

        if cstep % 100 == 0:
            save_grid(cgen, fixed_noise, SAMPLES_ROOT / 'cond' / f'step_{cstep:07d}.png', y=fixed_y)
            print(f'[cond] e{epoch} s{cstep} D={loss_d.item():.3f} G={loss_g.item():.3f} W={wasserstein.item():.3f}')

    cfid, cis_mean, cis_std, cn = eval_metrics_cond(cgen, cfid_m, cis_m)
    cepoch_metrics['epoch'].append(epoch)
    cepoch_metrics['fid'].append(cfid)
    cepoch_metrics['is_mean'].append(cis_mean)
    cepoch_metrics['is_std'].append(cis_std)
    print(f'[cond] epoch {epoch} FID={cfid:.2f} IS={cis_mean:.2f}±{cis_std:.2f} n={cn}')

    save_grid(cgen, fixed_noise, SAMPLES_ROOT / 'cond' / f'epoch_{epoch:03d}.png', y=fixed_y)
    tmp = cckpt_path.with_suffix('.tmp')
    torch.save({'epoch': epoch, 'step': cstep, 'gen': cgen.state_dict(),
                'disc': cdisc.state_dict(), 'opt_g': copt_g.state_dict(),
                'opt_d': copt_d.state_dict(), 'hist': chist,
                'epoch_metrics': cepoch_metrics}, tmp)
    tmp.replace(cckpt_path)

## Графики и итоговые метрики

In [ ]:
import matplotlib.pyplot as plt


def smooth_curve(arr, w=200):
    if len(arr) < w:
        return np.array(arr)
    return np.convolve(arr, np.ones(w) / w, mode='valid')


def plot_losses(hist_dict, title):
    keys = [('d', 'Discriminator'), ('g', 'Generator'), ('wass', 'Wasserstein'), ('gp', 'GP')]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (k, lbl) in zip(axes.flat, keys):
        ax.plot(smooth_curve(hist_dict[k]), linewidth=0.6)
        ax.set_title(lbl)
        ax.set_xlabel('step')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_epoch_metrics(m_dict, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(m_dict['epoch'], m_dict['fid'], 'o-', markersize=3)
    ax1.set_title('FID'); ax1.set_xlabel('epoch')
    ax2.plot(m_dict['epoch'], m_dict['is_mean'], 'o-', markersize=3)
    ax2.fill_between(m_dict['epoch'],
                     np.array(m_dict['is_mean']) - np.array(m_dict['is_std']),
                     np.array(m_dict['is_mean']) + np.array(m_dict['is_std']),
                     alpha=0.2)
    ax2.set_title('IS'); ax2.set_xlabel('epoch')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_losses(hist, 'Безусловная генерация')
plot_epoch_metrics(epoch_metrics, 'Безусловная генерация')

In [ ]:
plot_losses(chist, 'Условная генерация')
plot_epoch_metrics(cepoch_metrics, 'Условная генерация')

In [ ]:
gen.eval()
with torch.inference_mode():
    z = torch.randn(64, Z_DIM, 1, 1, device=DEVICE)
    imgs = (gen(z).cpu() * 0.5 + 0.5).clamp(0, 1)

grid = vutils.make_grid(imgs, nrow=8).permute(1, 2, 0).tolist()
plt.figure(figsize=(10, 10))
plt.imshow(grid)
plt.title('Безусловная генерация')
plt.axis('off')
plt.show()
gen.train()

In [ ]:
cgen.eval()
with torch.inference_mode():
    z = torch.randn(64, Z_DIM, 1, 1, device=DEVICE)
    y_vis = torch.tensor([0] * 32 + [1] * 32, device=DEVICE)
    imgs_c = (cgen(z, y_vis).cpu() * 0.5 + 0.5).clamp(0, 1)

grid_c = vutils.make_grid(imgs_c, nrow=8).permute(1, 2, 0).tolist()
plt.figure(figsize=(10, 10))
plt.imshow(grid_c)
plt.title('Условная генерация (0=female, 1=male)')
plt.axis('off')
plt.show()
cgen.train()

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

N_EVAL = 1024


def final_eval(generator, loader, n_eval=N_EVAL, n_classes=None):
    fid_eval = FrechetInceptionDistance(feature=2048, normalize=True).to('cpu')
    is_eval = InceptionScore(normalize=True, splits=10).to('cpu')

    n = 0
    for batch in loader:
        imgs = batch[0] if isinstance(batch, (tuple, list)) else batch
        fid_eval.update(clip_01(imgs).to('cpu', torch.float32), real=True)
        n += imgs.size(0)
        if n >= n_eval:
            break

    generator.eval()
    generated = 0
    while generated < n_eval:
        b = min(64, n_eval - generated)
        z = torch.randn(b, Z_DIM, 1, 1, device=DEVICE)
        if n_classes:
            y_r = torch.randint(0, n_classes, (b,), device=DEVICE)
            fake = clip_01(generator(z, y_r)).cpu()
        else:
            fake = clip_01(generator(z)).cpu()
        fid_eval.update(fake, real=False)
        is_eval.update(fake)
        generated += b

    fid = float(fid_eval.compute().item())
    is_mean, is_std = is_eval.compute()
    generator.train()
    return fid, float(is_mean.item()), float(is_std.item())


fid_u, is_u_mean, is_u_std = final_eval(gen, val_loader, n_eval=N_EVAL)
print(f'Безусловный WGAN-GP (n={N_EVAL}): FID={fid_u:.2f} IS={is_u_mean:.2f}±{is_u_std:.2f}')

fid_c, is_c_mean, is_c_std = final_eval(cgen, cond_val_loader, n_eval=N_EVAL, n_classes=2)
print(f'Условный WGAN-GP (n={N_EVAL}): FID={fid_c:.2f} IS={is_c_mean:.2f}±{is_c_std:.2f}')